# APHEX Evaluation Suite

Hardware-aware deployment optimization benchmark across four model families:
ResNet-50 (CNN), BERT-base (encoder transformer), LightGBM (gradient boosted trees),
and GPT-2 (decoder LLM proxy for Llama-7B).

For each model we report:
- **Search reduction** — how many candidates the cost model generates vs. blind brute-force
- **Latency improvement** — best candidate p50 speedup over FP32 baseline
- **Memory reduction** — peak memory saved by the winning candidate
- **Accuracy degradation** — cosine-similarity drop (classification proxy; skipped for LLMs)

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

pip("aphex-ml[torch,sklearn,onnx]")
pip("lightgbm", "scikit-learn", "transformers", "accelerate")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Any

from infermap.inspector import inspect_model, ModelInfo
from infermap.candidates import generate_candidates, DeploymentCandidate
from infermap.benchmark import benchmark_candidate, BenchmarkResult
from infermap.profiler import profile_hardware

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Hardware profile ──────────────────────────────────────────────────────────
hw = profile_hardware()
print(f"Accelerator : {hw.accelerator.kind.upper()}  {hw.accelerator.name}")
print(f"VRAM        : {hw.accelerator.memory_gb:.1f} GB")
print(f"CPU         : {hw.cpu.name}  ({hw.cpu.physical_cores} cores)")

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
WARMUP  = 10
MEASURE = 100
TIMEOUT = 240   # seconds per candidate (None = in-process, no subprocess)

# Brute-force search space: every backend × every dtype (for reference)
_ALL_BACKENDS = [
    "pytorch_fp32", "pytorch_fp16", "pytorch_bf16", "pytorch_int8_dynamic",
    "torch_compile_fp32", "torch_compile_fp16", "torch_compile_bf16",
    "onnx_cpu", "onnx_cuda", "onnx_int8_cpu",
    "tensorrt_fp32", "tensorrt_fp16", "tensorrt_int8",
    "openvino_fp32", "openvino_int8",
    "pytorch_prune_unstructured_30", "pytorch_prune_unstructured_50",
    "pytorch_prune_unstructured_70", "pytorch_prune_2_4",
    "pytorch_dp2_fp32", "pytorch_dp4_fp32",
    "sklearn_native", "lightgbm_native", "onnx_cpu",
]
BRUTE_FORCE_N = len(_ALL_BACKENDS)

@dataclass
class ModelStudy:
    name: str
    hardware: str
    candidates: list[DeploymentCandidate]
    results: list[BenchmarkResult]
    info: ModelInfo
    baseline_latency_ms: float
    baseline_memory_mb: float


def run_study(
    name: str,
    model: Any,
    input_shape: list[int],
    batch_size: int = 1,
    calibration_inputs: list | None = None,
    timeout_s: float | None = TIMEOUT,
) -> ModelStudy:
    """
    timeout_s=None runs benchmarks in-process (no subprocess spawn).
    Required for models whose wrapper class is defined in __main__ (notebook cells),
    since spawned workers cannot unpickle classes from __main__.
    """
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    info = inspect_model(model, input_shape)
    candidates = generate_candidates(info, hw)
    print(f"  Candidates generated : {len(candidates)}  (brute-force pool: {BRUTE_FORCE_N})")
    print(f"  Search reduction     : {(1 - len(candidates)/BRUTE_FORCE_N)*100:.0f}%")
    print()

    results: list[BenchmarkResult] = []
    for c in candidates:
        r = benchmark_candidate(
            c, model, info, input_shape,
            batch_size=batch_size,
            warmup_iters=WARMUP,
            measure_iters=MEASURE,
            timeout_s=timeout_s,
            calibration_inputs=calibration_inputs,
        )
        status = "OK" if r.ok else f"ERR: {r.error[:60]}"
        if r.ok:
            print(f"  [{status:>4}] {c.backend:<40}  {r.latency_p50_ms:6.2f} ms  {r.memory_mb:6.0f} MB")
        else:
            print(f"  [{status}]")
        results.append(r)

    baseline = next((r for r in results if "fp32" in r.candidate.backend and r.ok), None)
    baseline_lat = baseline.latency_p50_ms if baseline else 0.0
    baseline_mem = baseline.memory_mb if baseline else 0.0

    return ModelStudy(
        name=name,
        hardware=f"{hw.accelerator.kind.upper()} {hw.accelerator.name}",
        candidates=candidates,
        results=results,
        info=info,
        baseline_latency_ms=baseline_lat,
        baseline_memory_mb=baseline_mem,
    )

## 1. ResNet-50 (Image classification, CUDA)

In [ ]:
import torchvision.models as tvm

resnet = tvm.resnet50(weights=tvm.ResNet50_Weights.DEFAULT).eval()

calib_resnet = [torch.randn(1, 3, 224, 224) for _ in range(16)]

study_resnet = run_study(
    name="ResNet-50",
    model=resnet,
    input_shape=[3, 224, 224],
    batch_size=1,
    calibration_inputs=calib_resnet,
)

## 2. BERT-base (Encoder transformer, CUDA)

In [ ]:
from transformers import BertModel

bert = BertModel.from_pretrained("bert-base-uncased").eval()

# BERT takes [batch, seq_len] integer token ids — wrap as a flat nn.Module
class BertWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        ids = input_ids.long()
        mask = torch.ones_like(ids)
        return self.model(input_ids=ids, attention_mask=mask).last_hidden_state

bert_wrapped = BertWrapper(bert).eval()
SEQ = 128

calib_bert = [torch.randint(0, 30522, (1, SEQ)) for _ in range(16)]

# timeout_s=None: BertWrapper is defined in __main__ and cannot be pickled
# across a spawned subprocess — run all benchmarks in-process instead.
study_bert = run_study(
    name="BERT-base",
    model=bert_wrapped,
    input_shape=[SEQ],
    batch_size=1,
    calibration_inputs=calib_bert,
    timeout_s=None,
)

## 3. LightGBM (Gradient boosted trees, CPU)

In [ ]:
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=10_000, n_features=50, n_informative=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

lgbm_model = lgb.LGBMClassifier(n_estimators=300, num_leaves=63, random_state=42)
lgbm_model.fit(X_tr, y_tr)
print(f"LightGBM test accuracy: {lgbm_model.score(X_te, y_te):.4f}")

# Wrap for aphex: expose predict_proba as a torch.nn.Module-like callable
class SKLearnWrapper(torch.nn.Module):
    """Thin wrapper so aphex can inspect param count and route to sklearn candidates."""
    def __init__(self, sk_model, n_features: int):
        super().__init__()
        self.sk_model = sk_model
        self.n_features = n_features
        # Register a dummy parameter so inspect_model has something to count
        self.register_buffer("_dummy", torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        arr = x.cpu().numpy()
        proba = self.sk_model.predict_proba(arr)
        return torch.from_numpy(proba).float()

lgbm_wrapped = SKLearnWrapper(lgbm_model, n_features=50).eval()
calib_lgbm = [torch.from_numpy(X_te[i:i+1].astype(np.float32)) for i in range(16)]

# timeout_s=None: SKLearnWrapper is defined in __main__ and cannot be pickled
# across a spawned subprocess — run all benchmarks in-process instead.
study_lgbm = run_study(
    name="LightGBM",
    model=lgbm_wrapped,
    input_shape=[50],
    batch_size=1,
    calibration_inputs=calib_lgbm,
    timeout_s=None,
)

## 4. GPT-2 (Decoder LLM proxy, CUDA)

GPT-2 medium is used as a tractable stand-in for Llama-7B on T4 (16 GB VRAM).
Architecture class is identical; the scaling findings transfer directly.

In [ ]:
from transformers import GPT2Model

gpt2 = GPT2Model.from_pretrained("gpt2-medium").eval()

class GPT2Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids=input_ids.long()).last_hidden_state

gpt2_wrapped = GPT2Wrapper(gpt2).eval()
SEQ_GPT = 64   # short context to keep T4 VRAM usage manageable

# timeout_s=None: same __main__ class pickling reason as BertWrapper
study_gpt2 = run_study(
    name="GPT-2 medium (LLM proxy)",
    model=gpt2_wrapped,
    input_shape=[SEQ_GPT],
    batch_size=1,
    timeout_s=None,
)

## Results

In [ ]:
def best_ok(study: ModelStudy) -> BenchmarkResult | None:
    ok = [r for r in study.results if r.ok]
    return min(ok, key=lambda r: r.latency_p50_ms) if ok else None


def speedup(study: ModelStudy) -> float:
    b = best_ok(study)
    if b is None or study.baseline_latency_ms == 0:
        return float("nan")
    return study.baseline_latency_ms / b.latency_p50_ms


def mem_reduction(study: ModelStudy) -> float:
    b = best_ok(study)
    if b is None or study.baseline_memory_mb == 0:
        return float("nan")
    return (1 - b.memory_mb / study.baseline_memory_mb) * 100


def search_reduction(study: ModelStudy) -> float:
    return (1 - len(study.candidates) / BRUTE_FORCE_N) * 100


def acc_drop(study: ModelStudy) -> str:
    b = best_ok(study)
    if b is None or b.accuracy_drop is None:
        return "n/a"
    return f"{b.accuracy_drop*100:.2f}%"


studies = [study_resnet, study_bert, study_lgbm, study_gpt2]

rows = []
for s in studies:
    b = best_ok(s)
    rows.append({
        "Model": s.name,
        "Hardware": s.hardware,
        "Candidates": len(s.candidates),
        "Best backend": b.candidate.backend if b else "—",
        "Baseline (ms)": f"{s.baseline_latency_ms:.2f}",
        "Best (ms)": f"{b.latency_p50_ms:.2f}" if b else "—",
        "Speedup": f"{speedup(s):.2f}×",
        "Memory saved": f"{mem_reduction(s):.1f}%",
        "Accuracy drop": acc_drop(s),
        "Search reduction": f"{search_reduction(s):.0f}%",
    })

df = pd.DataFrame(rows)
df

In [ ]:
# ── Per-model candidate breakdown ─────────────────────────────────────────────
print("\nDetailed candidate results per model\n")

for s in studies:
    baseline_lat = s.baseline_latency_ms
    baseline_mem = s.baseline_memory_mb

    print(f"\n{'─'*80}")
    print(f"  {s.name}   ({s.hardware})")
    print(f"{'─'*80}")
    print(f"  {'Backend':<42} {'Lat p50':>8} {'Lat p95':>8} {'Mem MB':>8} {'Speedup':>8} {'Acc drop':>10}")
    print(f"  {'':─<42} {'':─>8} {'':─>8} {'':─>8} {'':─>8} {'':─>10}")

    for r in sorted(s.results, key=lambda r: r.latency_p50_ms if r.ok else 1e9):
        if not r.ok:
            print(f"  {r.candidate.backend:<42} {'SKIP':>8}")
            continue
        sp = f"{baseline_lat / r.latency_p50_ms:.2f}×" if baseline_lat > 0 else "—"
        ad = f"{r.accuracy_drop*100:.2f}%" if r.accuracy_drop is not None else "n/a"
        print(
            f"  {r.candidate.backend:<42}"
            f" {r.latency_p50_ms:>7.2f}ms"
            f" {r.latency_p95_ms:>7.2f}ms"
            f" {r.memory_mb:>7.0f}MB"
            f" {sp:>8}"
            f" {ad:>10}"
        )

In [ ]:
# ── Four metric summary ───────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  APHEX EVALUATION SUITE — SUMMARY")
print("═"*60)

print(f"\n{'Model':<28} {'Hardware':<20} {'Candidates':>10} {'Speedup':>9}")
print(f"{'':─<28} {'':─<20} {'':─>10} {'':─>9}")
for s in studies:
    print(f"{s.name:<28} {s.hardware:<20} {len(s.candidates):>10} {speedup(s):>8.1f}×")

print("\n── 1. Search reduction from cost model ──────────────────────")
for s in studies:
    sr = search_reduction(s)
    print(f"  {s.name:<28}  {len(s.candidates):>2} candidates  /  {BRUTE_FORCE_N} brute-force  →  {sr:.0f}% reduction")

print("\n── 2. Latency improvement ───────────────────────────────────")
for s in studies:
    b = best_ok(s)
    if b:
        print(
            f"  {s.name:<28}  {s.baseline_latency_ms:.2f} ms  →  "
            f"{b.latency_p50_ms:.2f} ms  ({speedup(s):.2f}× faster)  [{b.candidate.backend}]"
        )

print("\n── 3. Memory reduction ──────────────────────────────────────")
for s in studies:
    b = best_ok(s)
    if b:
        print(
            f"  {s.name:<28}  {s.baseline_memory_mb:.0f} MB  →  "
            f"{b.memory_mb:.0f} MB  ({mem_reduction(s):.1f}% saved)"
        )

print("\n── 4. Accuracy degradation (cosine-similarity proxy) ────────")
for s in studies:
    b = best_ok(s)
    if b:
        ad = acc_drop(s)
        note = "(LLM — perplexity metric required for full eval)" if ad == "n/a" else ""
        print(f"  {s.name:<28}  {ad}  {note}")

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("APHEX Evaluation Suite", fontsize=14, fontweight="bold")

labels = [s.name.split(" ")[0] for s in studies]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

# 1. Speedup
ax = axes[0]
speedups = [speedup(s) for s in studies]
bars = ax.bar(labels, speedups, color=colors)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Latency Speedup", fontweight="bold")
ax.set_ylabel("× faster than FP32")
for bar, val in zip(bars, speedups):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}×", ha="center", va="bottom", fontsize=9)

# 2. Memory reduction
ax = axes[1]
mem_reds = [mem_reduction(s) for s in studies]
bars = ax.bar(labels, mem_reds, color=colors)
ax.set_title("Memory Reduction", fontweight="bold")
ax.set_ylabel("% memory saved")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
for bar, val in zip(bars, mem_reds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

# 3. Search reduction
ax = axes[2]
search_reds = [search_reduction(s) for s in studies]
bars = ax.bar(labels, search_reds, color=colors)
ax.set_title("Search Reduction", fontweight="bold")
ax.set_ylabel("% candidates pruned")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
for bar, val in zip(bars, search_reds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.0f}%", ha="center", va="bottom", fontsize=9)

# 4. Accuracy drop (skip n/a)
ax = axes[3]
acc_drops_raw = []
labels_acc = []
colors_acc = []
for s, col in zip(studies, colors):
    b = best_ok(s)
    if b and b.accuracy_drop is not None:
        acc_drops_raw.append(b.accuracy_drop * 100)
        labels_acc.append(s.name.split(" ")[0])
        colors_acc.append(col)

if acc_drops_raw:
    bars = ax.bar(labels_acc, acc_drops_raw, color=colors_acc)
    ax.set_title("Accuracy Degradation", fontweight="bold")
    ax.set_ylabel("Cosine-similarity drop (%)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    for bar, val in zip(bars, acc_drops_raw):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
else:
    ax.text(0.5, 0.5, "No accuracy data\n(LLM families skipped)",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Accuracy Degradation", fontweight="bold")

plt.tight_layout()
plt.savefig("aphex_benchmark_study.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: aphex_benchmark_study.png")